In [ ]:
import pandas as pd

def build_combined_dataset():
    # Load source datasets — ensure CSVs are in the same working directory
    orders = pd.read_csv('olist_orders_dataset.csv')
    reviews = pd.read_csv('olist_order_reviews_dataset.csv')
    customers = pd.read_csv('olist_customers_dataset.csv')

    print(f"Orders loaded: {len(orders)}")
    print(f"Customers loaded: {len(customers)}")
    print(f"Reviews loaded: {len(reviews)}")

    # Convert timestamps and de-duplicate reviews by keeping the latest response per order
    reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])
    reviews_clean = (
        reviews
        .sort_values('review_answer_timestamp', ascending=False)
        .drop_duplicates(subset='order_id', keep='first')
    )
    print(f"Reviews after deduplication: {len(reviews_clean)}")

    # Join orders with reviews, then with customers
    orders_with_reviews = orders.merge(reviews_clean, how='left', on='order_id')
    combined = orders_with_reviews.merge(customers, how='left', on='customer_id')

    # Validation check
    print("-" * 40)
    print(f"Combined dataset rows: {len(combined)}")
    if len(combined) == len(orders):
        print("CHECK PASSED: Row count matches orders.")
    else:
        print(f"WARNING: Row count differs by {len(combined) - len(orders)} rows.")

    combined.to_csv('combined_dataset.csv', index=False)
    print("Saved as 'combined_dataset.csv'")

if __name__ == "__main__":
    build_combined_dataset()

In [ ]:
# Load the combined dataset and preview
df = pd.read_csv('combined_dataset.csv')
df.head()

In [ ]:
# Total number of columns in the combined dataset
print(f"Column count: {df.shape[1]}")

In [ ]:
# Inspect missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(missing[missing > 0].to_frame('missing_count').assign(pct=missing_pct))

In [ ]:
# Keep only orders with a confirmed delivery date
df_delivered = df.dropna(subset=['order_delivered_customer_date']).copy()
print(f"Rows retained after filtering undelivered orders: {len(df_delivered)}")

In [ ]:
# Confirm missing values in filtered dataset
df_delivered.isnull().sum()

In [ ]:
# Parse delivery date columns as datetime
for col in ['order_estimated_delivery_date', 'order_delivered_customer_date']:
    df_delivered[col] = pd.to_datetime(df_delivered[col])

In [ ]:
# Compute delivery buffer: positive = delivered early, negative = delivered late
df_delivered['Delivery_Buffer_Days'] = (
    df_delivered['order_estimated_delivery_date'] - df_delivered['order_delivered_customer_date']
).dt.days

In [ ]:
# Quick stats on delivery buffer
print(df_delivered['Delivery_Buffer_Days'].describe().round(2))

In [ ]:
# Classify delivery punctuality
def delivery_category(days):
    if days >= 0:
        return 'On Time'
    elif days >= -5:
        return 'Late'
    else:
        return 'Super Late'

df_delivered['Punctuality'] = df_delivered['Delivery_Buffer_Days'].apply(delivery_category)

In [ ]:
df_delivered.head()

In [ ]:
# Distribution of delivery punctuality categories
counts = df_delivered['Punctuality'].value_counts()
pcts = df_delivered['Punctuality'].value_counts(normalize=True).mul(100).round(2)
summary = counts.to_frame('count').assign(percentage=pcts)
print(summary)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
order = df_delivered['Punctuality'].value_counts().index
sns.countplot(y=df_delivered['Punctuality'], order=order, palette='Set2')
plt.title('Delivery Punctuality Distribution')
plt.xlabel('Number of Orders')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
# Unique customer states in the dataset
states = df_delivered['customer_state'].unique()
print(f"Unique states: {len(states)}")
print(states)

In [ ]:
# Flag whether order arrived after estimated delivery date
df_delivered['delivered_late'] = (
    df_delivered['order_delivered_customer_date'] > df_delivered['order_estimated_delivery_date']
)

In [ ]:
# Aggregate late delivery rate by state
state_stats = df_delivered.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    late_count=('delivered_late', 'sum')
).reset_index()

state_stats['late_rate'] = (state_stats['late_count'] / state_stats['total_orders']) * 100

In [ ]:
# Sort by late rate — best performers at top
state_ranked = state_stats.sort_values(by='late_rate', ascending=True)

In [ ]:
plt.figure(figsize=(10, 8))
plt.barh(state_ranked['customer_state'], state_ranked['late_rate'], color='steelblue')
plt.title('Late Delivery Rate by State (Best to Worst)')
plt.xlabel('Late Delivery Rate (%)')
plt.ylabel('State')
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.regplot(
    x='Delivery_Buffer_Days',
    y='review_score',
    data=df_delivered,
    scatter_kws={'alpha': 0.2, 'color': 'teal'},
    line_kws={'color': 'red'}
)
plt.title('Delivery Buffer vs Customer Review Score')
plt.xlabel('Delivery Buffer (Days, + = Early)')
plt.ylabel('Review Score (1–5)')
plt.tight_layout()
plt.show()

In [ ]:
# Average review score grouped by delivery buffer (rounded to nearest day)
df_delivered['Buffer_Day_Bucket'] = df_delivered['Delivery_Buffer_Days'].round()
score_by_buffer = (
    df_delivered
    .groupby('Buffer_Day_Bucket')['review_score']
    .agg(['mean', 'median', 'count'])
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=score_by_buffer, x='Buffer_Day_Bucket', y='mean', marker='s', label='Mean Score')
plt.axhline(y=score_by_buffer['mean'].mean(), color='gray', linestyle='--', label='Overall Mean')
plt.title('Average Review Score by Delivery Buffer (Days)')
plt.xlabel('Delivery Buffer (Days, + = Early)')
plt.ylabel('Avg Review Score (1–5)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Average review score by punctuality category
punctuality_review = (
    df_delivered
    .groupby('Punctuality')['review_score']
    .agg(['mean', 'std', 'count'])
    .round(2)
)
print(punctuality_review)

In [ ]:
# Load order items and product details
items_df = pd.read_csv('olist_order_items_dataset.csv')
products_df = pd.read_csv('olist_products_dataset.csv')
items_with_products = items_df.merge(products_df, how='left', on='product_id')

In [ ]:
# Attach product info to delivered orders
df_delivered = df_delivered.merge(items_with_products, how='left', on='order_id')

In [ ]:
# Load and attach English product category names
cat_trans = pd.read_csv('product_category_name_translation.csv')
df_delivered = df_delivered.merge(cat_trans, how='left', on='product_category_name')

# Verify
df_delivered[['product_category_name', 'product_category_name_english']].head()

In [ ]:
# Compute average review score per state
state_review = df_delivered.groupby('customer_state').agg(
    avg_review=('review_score', 'mean')
).reset_index()

# Join with state_stats for fuller picture
state_analysis = state_stats.merge(state_review, on='customer_state')
state_analysis = state_analysis.sort_values('avg_review', ascending=False)

plt.figure(figsize=(14, 6))
sns.barplot(data=state_analysis, x='customer_state', y='avg_review', palette='coolwarm')
plt.title('Average Review Score by State')
plt.xlabel('State')
plt.ylabel('Average Review Score')
plt.xticks(rotation=45)
plt.ylim(0, 5)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Identify first-time vs repeat customers and compute retention by punctuality
df_delivered = df_delivered.sort_values(by=['customer_unique_id', 'order_purchase_timestamp'])
df_delivered['visit_number'] = df_delivered.groupby('customer_unique_id').cumcount() + 1

# Flag customers who made more than one purchase
repeat_customers = df_delivered[df_delivered['visit_number'] > 1]['customer_unique_id'].unique()

# Focus on first visits
first_visits = df_delivered[df_delivered['visit_number'] == 1].copy()
first_visits['came_back'] = first_visits['customer_unique_id'].isin(repeat_customers)

retention = first_visits.groupby('Punctuality').agg(
    new_customers=('customer_unique_id', 'count'),
    returned=('came_back', 'sum')
)
retention['retention_pct'] = (retention['returned'] / retention['new_customers'] * 100).round(1)
print(retention)

retention['retention_pct'].plot(
    kind='bar',
    color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'],
    figsize=(9, 5),
    rot=30
)
plt.title('Customer Retention Rate by First Delivery Punctuality')
plt.ylabel('Retention Rate (%)')
plt.xlabel('Punctuality Category')
plt.tight_layout()
plt.show()